[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/04_llm_finetuning/full_finetuning.ipynb)

# 02 . Full Finetuning of a Tiny GPT

## Concept — Full finetuning (revision notes)

**What it is**
- Full finetuning continues training a pretrained model with **every** parameter trainable (no adapters, no freezing), on a smaller, task- or domain-specific dataset.

**Why it matters**
- It is the highest-capacity way to adapt a model and the baseline every parameter-efficient method (LoRA, adapters, prompt tuning) is compared against. Its cost is dominated by memory: weights + gradients + optimizer states + activations.

**When to use**
- You have enough GPU memory, a decent amount of data (thousands+ examples), and need maximum quality or a big behaviour/domain shift. Prefer LoRA when memory is tight or you need many task variants of one base model.

**Math & intuition**
- Same objective as pretraining: minimise next-token cross-entropy, but with a much smaller learning rate (typically 10-100x lower, e.g. `1e-5` to `5e-5`), warm-up and decay, and gradient clipping, so you adapt without destroying what was learned.
- Memory per parameter with Adam in fp32: weights 4 B + grads 4 B + `exp_avg` 4 B + `exp_avg_sq` 4 B = **16 B/param**. Mixed precision (bf16 weights/grads + fp32 master weights and Adam states) is also about 16 B/param (2 + 2 + 4 + 4 + 4). A 7B model therefore needs about 112 GB before activations.
- Activations scale with batch x sequence x layers. **Gradient checkpointing** stores only each block's input and recomputes the rest during backward: roughly 1/layers of the activation memory for about one extra forward pass of compute.
- **Catastrophic forgetting:** moving all weights toward the new domain can degrade old-domain performance. We measure this directly below.

### 1. Synthetic 'pretraining' and 'finetuning' domains

**What this cell does (plain language):** we generate two tiny synthetic text domains with a small grammar: domain A (animals) plays the role of the pretraining corpus and domain B (space words) is the new domain we will finetune on. Both share the same sentence structure and alphabet, so a model pretrained on A should transfer to B.

In [1]:
import random, time, math, copy, torch
random.seed(0)

A = dict(adj=["big", "small", "red", "quick"], noun=["cat", "dog", "bird", "fish"], verb=["sees", "likes", "chases"])
B = dict(adj=["old", "bright", "cold", "vast"], noun=["robot", "planet", "comet", "probe"], verb=["scans", "orbits", "finds"])

def make_text(g, n, rng):
    s = []
    for _ in range(n):
        s.append(f"the {rng.choice(g['adj'])} {rng.choice(g['noun'])} {rng.choice(g['verb'])} the {rng.choice(g['noun'])}.\n")
    return "".join(s)

rng = random.Random(0)
text_A, text_B = make_text(A, 600, rng), make_text(B, 600, rng)
text_A_val, text_B_val = make_text(A, 100, rng), make_text(B, 100, rng)
chars = sorted(set(text_A + text_B + text_A_val + text_B_val))
stoi = {c: i for i, c in enumerate(chars)}; itos = chars
enc = lambda s: torch.tensor([stoi[c] for c in s])
dec = lambda ids: "".join(itos[i] for i in ids)
dA, dB, vA, vB = map(enc, (text_A, text_B, text_A_val, text_B_val))
print("vocab size:", len(chars), "| tokens in A / B train:", len(dA), len(dB))
print("sample A:", repr(text_A[:45])); print("sample B:", repr(text_B[:48]))

vocab size: 24 | tokens in A / B train: 17362 19913
sample A: 'the quick fish sees the bird.\nthe quick fish '
sample B: 'the vast planet finds the probe.\nthe vast planet'


### 2. The model and a reusable training loop

**What this cell does (plain language):** we define the tiny GPT (same code as the other notebooks in this folder), a random-window batch sampler, an evaluation helper, and a `train()` function with the finetuning essentials: AdamW, linear warm-up + cosine decay, and gradient clipping at 1.0.

In [2]:
import math, torch, torch.utils.checkpoint, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass, asdict

@dataclass
class GPTConfig:
    vocab_size: int = 64
    block_size: int = 32
    n_layer: int = 2
    n_head: int = 4
    n_embd: int = 64

class Attn(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd)
        self.proj = nn.Linear(c.n_embd, c.n_embd)
        self.h = c.n_head
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # causal mask built in
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(c.n_embd), nn.LayerNorm(c.n_embd)
        self.attn = Attn(c)
        self.mlp = nn.Sequential(nn.Linear(c.n_embd, 4 * c.n_embd), nn.GELU(), nn.Linear(4 * c.n_embd, c.n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.cfg = c
        self.tok = nn.Embedding(c.vocab_size, c.n_embd)
        self.pos = nn.Embedding(c.block_size, c.n_embd)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd)
        self.head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.use_ckpt = False
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)       # small init => starting loss ~ ln(vocab)
            if getattr(m, 'bias', None) is not None: nn.init.zeros_(m.bias)
    def forward(self, idx, targets=None, ignore_index=-100):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for b in self.blocks:
            if self.use_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(b, x, use_reentrant=False)
            else:
                x = b(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=ignore_index)
        return logits, loss

BLOCK = 32
def get_batch(data, bs, gen):
    ix = torch.randint(0, len(data) - BLOCK - 1, (bs,), generator=gen)
    x = torch.stack([data[i:i + BLOCK] for i in ix]); y = torch.stack([data[i + 1:i + BLOCK + 1] for i in ix])
    return x, y

@torch.no_grad()
def evaluate(model, data, n=20):
    model.eval(); g = torch.Generator().manual_seed(123)
    ls = [model(*get_batch(data, 16, g))[1].item() for _ in range(n)]
    model.train(); return sum(ls) / n

def lr_at(step, total, peak, warmup=10):
    if step < warmup: return peak * (step + 1) / warmup                      # linear warm-up
    return 0.5 * peak * (1 + math.cos(math.pi * (step - warmup) / (total - warmup)))  # cosine decay

def train(model, data, steps, peak_lr, bs=16, seed=0, params=None, log_every=0, clip=1.0):
    opt = torch.optim.AdamW(params if params is not None else model.parameters(), lr=peak_lr, weight_decay=0.0)
    g = torch.Generator().manual_seed(seed); model.train(); hist = []
    for s in range(steps):
        for pg in opt.param_groups: pg["lr"] = lr_at(s, steps, peak_lr)
        x, y = get_batch(data, bs, g)
        _, loss = model(x, y)
        opt.zero_grad(); loss.backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), clip)    # returns the pre-clip norm
        opt.step(); hist.append(loss.item())
        if log_every and s % log_every == 0: print(f"  step {s:3d} loss {loss.item():.3f} grad-norm {gn:.2f}")
    return opt, hist

cfg = GPTConfig(vocab_size=len(chars), block_size=BLOCK, n_layer=2, n_head=4, n_embd=64)
torch.manual_seed(0)
base = GPT(cfg)
n_params = sum(p.numel() for p in base.parameters())
print("parameters:", n_params, "| initial loss on A:", round(evaluate(base, vA), 3), "(ln V =", round(math.log(len(chars)), 3), ")")

parameters: 103680 | initial loss on A: 3.214 (ln V = 3.178 )


### 3. 'Pretrain' the base model on domain A

**What this cell does (plain language):** we train from scratch on domain A for a few hundred steps and record validation loss on both domains. Domain A loss should fall well below the random baseline; domain B loss stays high because the model has never seen the space words.

In [3]:
t0 = time.time()
pre_opt, h = train(base, dA, steps=80, peak_lr=3e-3, log_every=30)
print(f"pretrain done in {time.time()-t0:.1f}s")
pre = {"A": evaluate(base, vA), "B": evaluate(base, vB)}
print(f"after pretraining  -> val loss A: {pre['A']:.3f} | val loss B: {pre['B']:.3f}")
base_state = copy.deepcopy(base.state_dict())    # keep a pristine copy of the 'pretrained checkpoint'

  step   0 loss 3.210 grad-norm 2.71


  step  30 loss 1.154 grad-norm 1.18


  step  60 loss 0.811 grad-norm 1.43


pretrain done in 2.5s
after pretraining  -> val loss A: 0.708 | val loss B: 3.346


### 4. Full-parameter finetuning on domain B (and the forgetting it causes)

**What this cell does (plain language):** we load the pretrained checkpoint, leave every parameter trainable (that is what makes it *full* finetuning), and train on domain B with a 10x smaller peak learning rate. We then compare val loss on B (should drop) and on A (typically rises: catastrophic forgetting).

In [4]:
ft = GPT(cfg); ft.load_state_dict(base_state)
print("trainable params:", sum(p.numel() for p in ft.parameters() if p.requires_grad), "of", n_params)
_, h_ft = train(ft, dB, steps=80, peak_lr=3e-4, log_every=20)
post = {"A": evaluate(ft, vA), "B": evaluate(ft, vB)}
print(f"before finetune -> A {pre['A']:.3f} | B {pre['B']:.3f}")
print(f"after  finetune -> A {post['A']:.3f} | B {post['B']:.3f}")
print("domain B improved:", post["B"] < pre["B"], "| domain A got worse (forgetting):", post["A"] > pre["A"])

trainable params: 103680 of 103680
  step   0 loss 3.385 grad-norm 3.87


  step  20 loss 2.002 grad-norm 1.98


  step  40 loss 1.480 grad-norm 1.27


  step  60 loss 1.327 grad-norm 1.09


before finetune -> A 0.708 | B 3.346
after  finetune -> A 1.832 | B 1.364
domain B improved: True | domain A got worse (forgetting): True


### 5. Is pretraining worth it? Finetune vs train-from-scratch

**What this cell does (plain language):** we train a freshly initialised model on domain B for the same 80 steps and compare with the pretrained-then-finetuned model. Both see identical finetuning data and step counts, so any gap comes from the transferred knowledge (alphabet, word shapes, sentence structure).

In [5]:
torch.manual_seed(0); scratch = GPT(cfg)
_, h_sc = train(scratch, dB, steps=80, peak_lr=3e-4)
sc_B = evaluate(scratch, vB)
print(f"from scratch, 80 steps @3e-4  -> val loss B {sc_B:.3f}")
print(f"pretrained + finetune          -> val loss B {post['B']:.3f}")
print("early training loss (avg of first 10 steps): scratch", round(sum(h_sc[:10])/10, 3), "| finetuned", round(sum(h_ft[:10])/10, 3))

from scratch, 80 steps @3e-4  -> val loss B 2.150
pretrained + finetune          -> val loss B 1.364
early training loss (avg of first 10 steps): scratch 3.104 | finetuned 3.105


### 6. Learning-rate sensitivity: too high destroys the pretrained weights

**What this cell does (plain language):** we finetune the same checkpoint at three peak learning rates and compare final domain-B loss and domain-A loss. A learning rate typical of pretraining (3e-3) churns the weights more, so the previously learned domain A is damaged more; a very small rate barely moves anything within the budget.

In [6]:
rows = []
for lr in (3e-5, 3e-4, 3e-3):
    m = GPT(cfg); m.load_state_dict(base_state)
    train(m, dB, steps=80, peak_lr=lr)
    rows.append((lr, evaluate(m, vB), evaluate(m, vA)))
print(f"{'peak lr':>8} | val B | val A (forgetting)")
for lr, b, a in rows: print(f"{lr:>8.0e} | {b:.3f} | {a:.3f}")

 peak lr | val B | val A (forgetting)
   3e-05 | 2.777 | 0.864
   3e-04 | 1.364 | 1.832
   3e-03 | 0.554 | 3.146


### 7. Memory math: parameters, gradients, Adam states

**What this cell does (plain language):** we count what a training step really stores for our tiny model by measuring the actual optimizer state tensors, and compare it with the 16-bytes-per-parameter rule. Then we use a small calculator to extrapolate to 125M, 1.3B, 7B and 70B parameters under different precision recipes.

In [7]:
def tensor_bytes(ts): return sum(t.numel() * t.element_size() for t in ts)
w_bytes = tensor_bytes(ft.parameters())
g_bytes = tensor_bytes(p.grad for p in ft.parameters() if p.grad is not None)
opt, _ = train(copy.deepcopy(ft), dB, steps=2, peak_lr=1e-4)      # a few steps so Adam has state
st = [t for s in opt.state.values() for k, t in s.items() if k in ("exp_avg", "exp_avg_sq")]
o_bytes = tensor_bytes(st)
print(f"weights {w_bytes/1e3:.0f} KB | grads {g_bytes/1e3:.0f} KB | adam states {o_bytes/1e3:.0f} KB")
print(f"total / params = {(w_bytes+g_bytes+o_bytes)/n_params:.1f} bytes per parameter (rule: 16)")

def train_mem_gb(n_params, recipe="fp32"):
    # bytes per parameter: (weights, grads, adam m+v, fp32 master copy)
    bpp = {"fp32": (4, 4, 8, 0), "bf16-mixed": (2, 2, 8, 4), "bf16-pure+8bit-adam": (2, 2, 2, 0)}[recipe]
    return n_params * sum(bpp) / 1e9
print(f"{'model':>6} | " + " | ".join(f"{r:>20}" for r in ("fp32", "bf16-mixed", "bf16-pure+8bit-adam")) + "   (GB, excl. activations)")
for name, n in [("125M", 125e6), ("1.3B", 1.3e9), ("7B", 7e9), ("70B", 70e9)]:
    print(f"{name:>6} | " + " | ".join(f"{train_mem_gb(n, r):>20.1f}" for r in ("fp32", "bf16-mixed", "bf16-pure+8bit-adam")))

weights 415 KB | grads 415 KB | adam states 829 KB
total / params = 16.0 bytes per parameter (rule: 16)
 model |                 fp32 |           bf16-mixed |  bf16-pure+8bit-adam   (GB, excl. activations)
  125M |                  2.0 |                  2.0 |                  0.8
  1.3B |                 20.8 |                 20.8 |                  7.8
    7B |                112.0 |                112.0 |                 42.0
   70B |               1120.0 |               1120.0 |                420.0


### 8. Gradient checkpointing: trade compute for activation memory

**What this cell does (plain language):** we measure how many bytes autograd keeps alive for the backward pass (using `saved_tensors_hooks`) on a deeper model with and without `torch.utils.checkpoint` around each block. We also verify the gradients are identical and time both. This counts tensors packed for backward, so it is a proxy (it includes saved weights) rather than an exact allocator reading.

In [8]:
torch.manual_seed(0)
deep = GPT(GPTConfig(vocab_size=len(chars), block_size=64, n_layer=8, n_head=4, n_embd=128))
xb = torch.randint(0, len(chars), (8, 64))

def saved_bytes_and_grads(ckpt):
    deep.use_ckpt = ckpt; deep.zero_grad(); total = [0]
    def pack(t): total[0] += t.numel() * t.element_size(); return t
    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        loss = deep(xb, xb)[1]
    loss.backward()
    ts = []
    for _ in range(3):                      # timing: best of 3 (shared CPUs are noisy)
        t0 = time.time(); deep.zero_grad(); deep(xb, xb)[1].backward(); ts.append(time.time() - t0)
    return total[0], deep.head.weight.grad.clone(), deep.blocks[0].mlp[0].weight.grad.clone(), min(ts)

b0, g0a, g0b, t_plain = saved_bytes_and_grads(False)
b1, g1a, g1b, t_ckpt = saved_bytes_and_grads(True)
print(f"saved for backward: no-ckpt {b0/1e6:.1f} MB | ckpt {b1/1e6:.1f} MB | reduction {b0/b1:.1f}x")
print(f"fwd+bwd time      : no-ckpt {t_plain*1e3:.0f} ms | ckpt {t_ckpt*1e3:.0f} ms (checkpointing re-runs each block's forward during backward, so expect a modest slowdown, roughly 1.2-1.4x on an idle CPU; timings here are noisy)")
print("gradients identical:", torch.allclose(g0a, g1a, atol=1e-6), torch.allclose(g0b, g1b, atol=1e-6))

saved for backward: no-ckpt 42.7 MB | ckpt 2.7 MB | reduction 15.6x
fwd+bwd time      : no-ckpt 65 ms | ckpt 68 ms (checkpointing re-runs each block's forward during backward, so expect a modest slowdown, roughly 1.2-1.4x on an idle CPU; timings here are noisy)
gradients identical: True True


### 9. Partial finetuning: freezing layers to save memory

**What this cell does (plain language):** we freeze the embeddings and first block and train only the last block + final norm, which removes gradients and Adam states for most parameters. We compare the trainable fraction, the optimizer-state bytes, and the domain-B loss with full finetuning.

In [9]:
part = GPT(cfg); part.load_state_dict(base_state)
for p in part.parameters(): p.requires_grad_(False)
for p in list(part.blocks[-1].parameters()) + list(part.ln_f.parameters()): p.requires_grad_(True)
trainable = [p for p in part.parameters() if p.requires_grad]
n_tr = sum(p.numel() for p in trainable)
opt_p, _ = train(part, dB, steps=80, peak_lr=3e-4, params=trainable)
adam_bytes = tensor_bytes(t for s in opt_p.state.values() for k, t in s.items() if k in ("exp_avg", "exp_avg_sq"))
print(f"trainable {n_tr}/{n_params} = {100*n_tr/n_params:.0f}% | adam state bytes {adam_bytes/1e3:.0f} KB (full: {o_bytes/1e3:.0f} KB)")
print(f"val B: partial {evaluate(part, vB):.3f} | full {post['B']:.3f} | val A (forgetting): partial {evaluate(part, vA):.3f} | full {post['A']:.3f}")
print("note: tied embedding/head stayed frozen, so the head cannot learn new characters' statistics as freely.")

trainable 50112/103680 = 48% | adam state bytes 401 KB (full: 829 KB)
val B: partial 1.966 | full 1.364 | val A (forgetting): partial 1.246 | full 1.832
note: tied embedding/head stayed frozen, so the head cannot learn new characters' statistics as freely.


### 10. Checkpointing the run: weights vs weights + optimizer state

**What this cell does (plain language):** we save a 'weights only' file (for inference/sharing) and a 'full training checkpoint' (weights + Adam states + step) to see why resumable checkpoints are about 3x bigger, then confirm that resuming gives exactly the same next step as continuing without interruption.

In [10]:
import os, tempfile
with tempfile.TemporaryDirectory() as d:
    torch.save(ft.state_dict(), f"{d}/weights.pt")
    torch.save({"model": ft.state_dict(), "opt": opt.state_dict(), "step": 2}, f"{d}/full.pt")
    sw, sf = os.path.getsize(f"{d}/weights.pt"), os.path.getsize(f"{d}/full.pt")
print(f"weights-only {sw/1e3:.0f} KB | full training checkpoint {sf/1e3:.0f} KB | ratio {sf/sw:.2f}x")

# Resume test: continue 1 step directly vs. via a checkpoint round-trip -> identical weights.
def one_step(m, o, x, y):
    o.zero_grad(); m(x, y)[1].backward(); o.step()
mA = GPT(cfg); mA.load_state_dict(base_state); oA = torch.optim.AdamW(mA.parameters(), lr=1e-4)
x, y = get_batch(dB, 8, torch.Generator().manual_seed(5))
one_step(mA, oA, x, y)
opt_sd = copy.deepcopy(oA.state_dict())   # snapshot: load_state_dict can ALIAS tensors, so copy before A keeps stepping
mB = GPT(cfg); mB.load_state_dict(mA.state_dict()); oB = torch.optim.AdamW(mB.parameters(), lr=1e-4); oB.load_state_dict(opt_sd)
mC = GPT(cfg); mC.load_state_dict(mA.state_dict()); oC = torch.optim.AdamW(mC.parameters(), lr=1e-4)   # forgot optimizer state!
one_step(mA, oA, x, y); one_step(mB, oB, x, y); one_step(mC, oC, x, y)
same = lambda a, b: max((p - q).abs().max().item() for p, q in zip(a.parameters(), b.parameters()))
print("resume WITH optimizer state, max weight diff:   ", same(mA, mB))
print("resume WITHOUT optimizer state, max weight diff:", same(mA, mC))

weights-only 424 KB | full training checkpoint 1271 KB | ratio 3.00x
resume WITH optimizer state, max weight diff:    0.0
resume WITHOUT optimizer state, max weight diff: 0.0001667216420173645


## Common bugs

- **Using the pretraining learning rate.** `3e-3`-ish rates wreck a pretrained model (see section 6). Fix: start at 1e-5 to 5e-5 for real LLMs, with warm-up and decay.
- **Forgetting `model.train()` / `model.eval()`.** Dropout and other train-time behaviours stay in the wrong mode during eval or training. Fix: call `train()` before the loop and `eval()` inside the evaluation helper (and switch back).
- **No gradient clipping.** A single bad batch can spike the gradient norm and diverge the run. Fix: `clip_grad_norm_(params, 1.0)` after `backward` and before `step`; log the returned norm.
- **Evaluating only on the new domain.** You will not notice forgetting. Fix: always keep a small held-out set of the original behaviour/domain (or a general benchmark) and track it too.
- **Resuming without optimizer state.** AdamW's moment estimates are part of training state; restarting them changes the trajectory (section 10). Also `optimizer.load_state_dict` may alias tensors from the dict you pass, so deep-copy a snapshot you intend to reuse. Fix: save/load `optimizer.state_dict()` plus the scheduler and RNG/step counters.
- **Underestimating memory.** Counting only weights misses gradients, Adam states, fp32 master weights and activations (a 7B model is about 112 GB for weights+grads+Adam, not 14 GB). Fix: use the 16 B/param rule plus an activation estimate.
- **Gradient checkpointing with `use_reentrant=True` and frozen inputs.** The reentrant variant needs at least one input requiring grad or gradients silently vanish. Fix: pass `use_reentrant=False` (the recommended default).
- **Weight decay on biases and norms.** Decaying LayerNorm/bias parameters hurts. Fix: use two param groups, with `weight_decay=0` for 1-D tensors.
- **Tied embeddings and freezing.** Freezing the embedding also freezes the tied output head. Fix: untie them, or unfreeze deliberately.

## Exercises

**Exercise 1.** Print the number of parameters per component (embeddings, each block's attention, each block's MLP, final norm). What fraction is the (tied) embedding table?

In [11]:
# Your attempt for Exercise 1 here

**Solution 1**

In [12]:
# Solution 1
emb = ft.tok.weight.numel()
print("embedding (tied with head):", emb)
for i, b in enumerate(ft.blocks):
    print(f"block {i}: attn {sum(p.numel() for p in b.attn.parameters())} | mlp {sum(p.numel() for p in b.mlp.parameters())}")
print("fraction embedding:", emb / n_params)

embedding (tied with head): 1536
block 0: attn 16640 | mlp 33088
block 1: attn 16640 | mlp 33088
fraction embedding: 0.014814814814814815


**Exercise 2.** Add gradient accumulation (4 micro-batches of 4) to `train` and show the loss curve is similar to batch 16 with 1 micro-batch.

In [13]:
# Your attempt for Exercise 2 here

**Solution 2**

In [14]:
# Solution 2
def train_accum(model, data, steps, peak_lr, micro=4, accum=4, seed=0):
    opt = torch.optim.AdamW(model.parameters(), lr=peak_lr); g = torch.Generator().manual_seed(seed); hist = []
    for s in range(steps):
        for pg in opt.param_groups: pg["lr"] = lr_at(s, steps, peak_lr)
        opt.zero_grad(); tot = 0.0
        for _ in range(accum):
            x, y = get_batch(data, micro, g); l = model(x, y)[1] / accum; l.backward(); tot += l.item()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); hist.append(tot)
    return hist
m = GPT(cfg); m.load_state_dict(base_state)
h = train_accum(m, dB, 60, 3e-4)
print("loss first/last:", round(h[0], 3), round(h[-1], 3), "| val B:", round(evaluate(m, vB), 3))

loss first/last: 3.385 1.563 | val B: 1.535


**Exercise 3.** Mitigate forgetting by mixing 30% domain-A data into the finetuning set. Compare val A and val B with plain finetuning.

In [15]:
# Your attempt for Exercise 3 here

**Solution 3**

In [16]:
# Solution 3
mix = torch.cat([dB[:int(len(dB) * 0.7)], dA[:int(len(dB) * 0.3)]])
m = GPT(cfg); m.load_state_dict(base_state); train(m, mix, 80, 3e-4)
print(f"mixed: A {evaluate(m, vA):.3f} B {evaluate(m, vB):.3f} | plain: A {post['A']:.3f} B {post['B']:.3f}")

mixed: A 1.150 B 1.463 | plain: A 1.832 B 1.364


**Exercise 4.** Write a function that estimates activation memory for batch B, sequence T, layers L, hidden d (assume about 34*B*T*d bytes per layer in 16-bit, ignoring the T^2 term), and compare with checkpointing (only layer inputs stored: 2*B*T*d per layer).

In [17]:
# Your attempt for Exercise 4 here

**Solution 4**

In [18]:
# Solution 4
def act_gb(B, T, L, d, ckpt=False):
    per_layer = 2 * B * T * d if ckpt else 34 * B * T * d
    return L * per_layer / 1e9
print("7B-like (L=32, d=4096), B=8, T=2048:  plain %.1f GB | ckpt %.1f GB" % (act_gb(8, 2048, 32, 4096), act_gb(8, 2048, 32, 4096, True)))

7B-like (L=32, d=4096), B=8, T=2048:  plain 73.0 GB | ckpt 4.3 GB


**Exercise 5.** Create two AdamW param groups so biases and LayerNorm weights get `weight_decay=0` and other matrices get 0.1. Verify the group sizes.

In [19]:
# Your attempt for Exercise 5 here

**Solution 5**

In [20]:
# Solution 5
decay = [p for n, p in ft.named_parameters() if p.dim() >= 2]
no_decay = [p for n, p in ft.named_parameters() if p.dim() < 2]
o = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1}, {"params": no_decay, "weight_decay": 0.0}], lr=1e-4)
print(len(decay), "decayed tensors,", len(no_decay), "non-decayed tensors")

10 decayed tensors, 18 non-decayed tensors


**Exercise 6.** Measure the wall-clock overhead of gradient checkpointing for 1, 4 and 8 layers. Is the relative overhead roughly constant with depth? (timings on a shared CPU are noisy, take the best of several runs)

In [21]:
# Your attempt for Exercise 6 here

**Solution 6**

In [22]:
# Solution 6
for L in (1, 4, 8):
    mm = GPT(GPTConfig(vocab_size=len(chars), block_size=64, n_layer=L, n_head=4, n_embd=128)); res = []
    for ck in (False, True):
        mm.use_ckpt = ck; t0 = time.time()
        mm.zero_grad(); mm(xb, xb)[1].backward()      # untimed warm-up run
        ts = []
        for _ in range(4):
            t0 = time.time(); mm.zero_grad(); mm(xb, xb)[1].backward(); ts.append(time.time() - t0)
        res.append(min(ts) * 1e3)
    print(f"L={L}: plain {res[0]:.0f} ms | ckpt {res[1]:.0f} ms | overhead x{res[1]/res[0]:.2f}")

L=1: plain 7 ms | ckpt 9 ms | overhead x1.29


L=4: plain 24 ms | ckpt 38 ms | overhead x1.56


L=8: plain 66 ms | ckpt 68 ms | overhead x1.03
